## add memory and dependencies

In [3]:
import os
import json
import uuid

from openai import OpenAI
import rich

from dotenv import load_dotenv
from zep_cloud.client import Zep
from zep_cloud import Message
from typing import Union, List, Dict, Optional


load_dotenv()

bot_name='amadeus'
zep = Zep(api_key=os.environ.get("ZEP_API_KEY"))

oai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_BASE_URL"),
)

user_name = "Alex"
user_id = user_name + str(uuid.uuid4())[:4]
session_id = str(uuid.uuid4())

zep.user.add(
    user_id=user_id,
    email=f"{user_name}@painters.com",
    first_name=user_name,
    last_name="Painter",
)

zep.memory.add_session(
    user_id=user_id,
    session_id=session_id,
)


support_cases = [
    {
        "subject": "Bug: Magic Pen Tool Drawing Goats Instead of Boats",
        "messages": [
            {
                "role": "user",
                "content": "Whenever I use the magic pen tool to draw boats, it ends up drawing goats instead.",
                "timestamp": "2024-03-16T14:20:00Z",
            },
            {
                "role": "support_agent",
                "content": f"Hi {user_name}, that sounds like a bug! Thanks for reporting it. Could you let me know exactly how you're using the tool when this happens?",
                "timestamp": "2024-03-16T14:22:00Z",
            },
            {
                "role": "user",
                "content": "Sure, I select the magic pen, draw a boat shape, and it just replaces the shape with goats.",
                "timestamp": "2024-03-16T14:25:00Z",
            },
            {
                "role": "support_agent",
                "content": "Got it! We'll escalate this to our engineering team. In the meantime, you can manually select the boat shape from the options rather than drawing it with the pen.",
                "timestamp": "2024-03-16T14:27:00Z",
            },
            {
                "role": "user",
                "content": "Okay, thanks. I hope it gets fixed soon!",
                "timestamp": "2024-03-16T14:30:00Z",
            },
        ],
        "status": "escalated",
    },
]

chat_history = [
    {
        "role": "assistant",
        "name": bot_name,
        "content": f"Hello {user_name}, welcome to PaintWiz support. How can I assist you today?",
        "timestamp": "2024-03-15T10:00:00Z",
    },
    {
        "role": "user",
        "name": user_name,
        "content": "I'm absolutely furious! Your AI art generation is completely broken!",
        "timestamp": "2024-03-15T10:02:00Z",
    },
    {
        "role": "assistant",
        "name": bot_name,
        "content": f"I'm sorry to hear that you're experiencing issues, {user_name}. Can you please provide more details about what's going wrong?",
        "timestamp": "2024-03-15T10:03:00Z",
    },
    {
        "role": "user",
        "name": user_name,
        "content": "Every time I try to draw mountains, your stupid app keeps turning them into fountains! And what's worse, all the people in my drawings have six fingers! It's ridiculous!",
        "timestamp": "2024-03-15T10:05:00Z",
    },
    {
        "role": "assistant",
        "name": bot_name,
        "content": f"I sincerely apologize for the frustration this is causing you, {user_name}. That certainly sounds like a significant glitch in our system. I understand how disruptive this can be to your artistic process. Can you tell me which specific tool or feature you're using when this occurs?",
        "timestamp": "2024-03-15T10:06:00Z",
    },
    {
        "role": "user",
        "name": user_name,
        "content": "I'm using the landscape generator and the character creator. Both are completely messed up. How could you let this happen?",
        "timestamp": "2024-03-15T10:08:00Z",
    },
]

transactions = [
    {
        "date": "2024-07-30",
        "amount": 99.99,
        "status": "Success",
        "account_id": user_id,
        "card_last_four": "1234",
    },
    {
        "date": "2024-08-30",
        "amount": 99.99,
        "status": "Failed",
        "account_id": user_id,
        "card_last_four": "1234",
        "failure_reason": "Card expired",
    },
    {
        "date": "2024-09-15",
        "amount": 99.99,
        "status": "Failed",
        "account_id": user_id,
        "card_last_four": "1234",
        "failure_reason": "Card expired",
    },
]

account_status = {
    "user_id": user_id,
    "account": {
        "account_id": user_id,
        "account_status": {
            "status": "suspended",
            "reason": "payment failure",
        },
    },
}

def convert_to_zep_messages(chat_history: List[Dict[str, Union[str, None]]]) -> List[Message]:
    """
    Convert chat history to Zep messages.

    Args:
    chat_history (list): List of dictionaries containing chat messages.

    Returns:
    list: List of Zep Message objects.
    """
    return [
        Message(
            role_type=msg["role"],
            role=msg.get("name", None),
            content=msg["content"],
        )
        for msg in chat_history
    ]

# Zep's high-level API allows us to add a list of messages to a session.
zep.memory.add(
    session_id=session_id, messages=convert_to_zep_messages(chat_history)
)

# The lower-level data API allows us to add arbitrary data to a user's Knowledge Graph.
for tx in transactions:
    zep.graph.add(user_id=user_id, data=json.dumps(tx), type="json")

    zep.graph.add(
        user_id=user_id, data=json.dumps(account_status), type="json"
    )

for case in support_cases:
    zep.graph.add(user_id=user_id, data=json.dumps(case), type="json")

## test

In [4]:
all_user_edges = zep.graph.edge.get_by_user_id(user_id=user_id)
rich.print(all_user_edges[:3])


[
    EntityEdge(
        created_at='2025-03-17T03:28:11.159569Z',
        episodes=[],
        expired_at=None,
        fact='user has the email of Alex@painters.com',
        invalid_at=None,
        name='HAS_USER_EMAIL',
        source_node_uuid='52871b9b-ca5b-4b0a-865b-499789380b79',
        target_node_uuid='690e1ec5-579b-49ae-9bc5-9afdc3991ff3',
        uuid_='fdae80b9-a665-4456-8b87-24c6617e53cf',
        valid_at='2025-03-17T03:28:11.159569Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    ),
    EntityEdge(
        created_at='2025-03-17T03:28:11.159569Z',
        episodes=[],
        expired_at=None,
        fact='user has the name of Alex Painter',
        invalid_at=None,
        name='HAS_USER_NAME',
        source_node_uuid='52871b9b-ca5b-4b0a-865b-499789380b79',
        target_node_uuid='c01bdfd8-70aa-411d-8065-8d23f98907e7',
        uuid_='d1f7b1cb-596c-41aa-99cd-3e3e3617fd7f',
        valid_at='2025-03-17T03:28:11.159569Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    ),
    EntityEdge(
        created_at='2025-03-17T03:28:11.159569Z',
        episodes=[],
        expired_at=None,
        fact='user has the id of Alex1800',
        invalid_at=None,
        name='HAS_USER',
        source_node_uuid='52871b9b-ca5b-4b0a-865b-499789380b79',
        target_node_uuid='924248ae-6fd2-44f2-ba84-88db15211907',
        uuid_='4d991d72-021a-4dba-8f61-998cc0ba52e8',
        valid_at='2025-03-17T03:28:11.159569Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    )
]

In [5]:
memory = zep.memory.get(session_id=session_id)
rich.print(memory.context)


FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
<FACTS>
  - user has the name of Alex Painter (2025-03-17 03:28:11 - present)
  - user has the email of Alex@painters.com (2025-03-17 03:28:11 - present)
  - user has the id of Alex1800 (2025-03-17 03:28:11 - present)
</FACTS>

# These are the most relevant entities
# ENTITY_NAME: entity summary
<ENTITIES>
  - Alex@painters.com: user with the email of Alex@painters.com
  - Alex Painter: user with the name of Alex Painter
  - User: user
  - Alex1800: user with the id of Alex1800
</ENTITIES>

In [6]:
rich.print(memory.messages)


[
    Message(
        content='Hello Alex, welcome to PaintWiz support. How can I assist you today?',
        created_at='2025-03-17T03:28:10.688387Z',
        metadata=None,
        role='amadeus',
        role_type='assistant',
        token_count=0,
        updated_at='0001-01-01T00:00:00Z',
        uuid_='29796bb6-7380-4820-a726-968770d30608'
    ),
    Message(
        content="I'm absolutely furious! Your AI art generation is completely broken!",
        created_at='2025-03-17T03:28:10.688387Z',
        metadata=None,
        role='Alex',
        role_type='user',
        token_count=0,
        updated_at='0001-01-01T00:00:00Z',
        uuid_='d9eaf8ab-0ae3-45e3-9ea3-1d2d5bf33ca2'
    ),
    Message(
        content="I'm sorry to hear that you're experiencing issues, Alex. Can you please provide more details about
what's going wrong?",
        created_at='2025-03-17T03:28:10.688387Z',
        metadata=None,
        role='amadeus',
        role_type='assistant',
        token_count=0,
        updated_at='0001-01-01T00:00:00Z',
        uuid_='179818da-f047-4b8d-b570-25e67a1ad7af'
    ),
    Message(
        content="Every time I try to draw mountains, your stupid app keeps turning them into fountains! And what's 
worse, all the people in my drawings have six fingers! It's ridiculous!",
        created_at='2025-03-17T03:28:10.688387Z',
        metadata=None,
        role='Alex',
        role_type='user',
        token_count=0,
        updated_at='0001-01-01T00:00:00Z',
        uuid_='a853bd62-9f7c-4f85-9076-2a0d82741d1a'
    ),
    Message(
        content="I sincerely apologize for the frustration this is causing you, Alex. That certainly sounds like a 
significant glitch in our system. I understand how disruptive this can be to your artistic process. Can you tell me
which specific tool or feature you're using when this occurs?",
        created_at='2025-03-17T03:28:10.688387Z',
        metadata=None,
        role='amadeus',
        role_type='assistant',
        token_count=0,
        updated_at='0001-01-01T00:00:00Z',
        uuid_='a93202a3-191f-4b03-9b18-56737e2a86ee'
    ),
    Message(
        content="I'm using the landscape generator and the character creator. Both are completely messed up. How 
could you let this happen?",
        created_at='2025-03-17T03:28:10.688387Z',
        metadata=None,
        role='Alex',
        role_type='user',
        token_count=0,
        updated_at='0001-01-01T00:00:00Z',
        uuid_='540abc03-bd62-428b-b1f8-7b31819ee12b'
    )
]

In [7]:
r = zep.graph.search(user_id=user_id, query="Why are there so many goats?", limit=4, scope="edges")
rich.print(r.edges)


[
    EntityEdge(
        created_at='2025-03-17T03:28:36.968056Z',
        episodes=['d9eaf8ab-0ae3-45e3-9ea3-1d2d5bf33ca2'],
        expired_at=None,
        fact='Alex1800 is furious about the AI Art Generation, considering it to be completely broken.',
        invalid_at=None,
        name='IS_FURIOUS_ABOUT',
        source_node_uuid='52871b9b-ca5b-4b0a-865b-499789380b79',
        target_node_uuid='407cbbb2-5de6-4474-8342-5c58d5ccf7a8',
        uuid_='a8a24369-4366-4c22-816d-4fba6cb7dcd6',
        valid_at='2025-03-17T03:28:10.688387Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    ),
    EntityEdge(
        created_at='2025-03-17T03:28:26.221267Z',
        episodes=['29796bb6-7380-4820-a726-968770d30608'],
        expired_at=None,
        fact='Amadeus, the assistant, is providing support to Alex.',
        invalid_at=None,
        name='PROVIDES_SUPPORT_TO',
        source_node_uuid='097075c7-dd4f-4cfd-b42a-f52ee53fda5c',
        target_node_uuid='c01bdfd8-70aa-411d-8065-8d23f98907e7',
        uuid_='5abf104d-bd43-4f24-97b7-2f561039f81b',
        valid_at='2025-03-17T03:28:10.688387Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    ),
    EntityEdge(
        created_at='2025-03-17T03:28:26.221226Z',
        episodes=['29796bb6-7380-4820-a726-968770d30608'],
        expired_at=None,
        fact='Alex is seeking assistance from PaintWiz support.',
        invalid_at=None,
        name='SEEKS_SUPPORT_FROM',
        source_node_uuid='c01bdfd8-70aa-411d-8065-8d23f98907e7',
        target_node_uuid='c27c7b77-2574-4433-8119-8ab6cff5c6d0',
        uuid_='a167dd33-5080-46c3-ad8d-83863b408219',
        valid_at='2025-03-17T03:28:10.688387Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    ),
    EntityEdge(
        created_at='2025-03-17T03:28:11.159569Z',
        episodes=[],
        expired_at=None,
        fact='user has the id of Alex1800',
        invalid_at=None,
        name='HAS_USER',
        source_node_uuid='52871b9b-ca5b-4b0a-865b-499789380b79',
        target_node_uuid='924248ae-6fd2-44f2-ba84-88db15211907',
        uuid_='4d991d72-021a-4dba-8f61-998cc0ba52e8',
        valid_at='2025-03-17T03:28:11.159569Z',
        graph_id='52871b9b-ca5b-4b0a-865b-499789380b79'
    )
]

## chatbot

1.  first query the user message into zep.momery to check the memory
2.  pack the memory into the user message and query the gpt. put the momery as an assistant role. 
3.  get the gpt's output aligned with the memory!

In [8]:
new_session_id = str(uuid.uuid4())

alex_message = "Hi, I can't log in!"

# We start a new session indicating that Emily has started a new chat with the support agent.
zep.memory.add_session(user_id=user_id, session_id=new_session_id)

# We need to add the Emily's message to the session in order for memory.get to return
# relevant facts related to the message
zep.memory.add(
    session_id=new_session_id,
    messages=[Message(role_type="user", role=user_name, content=alex_message)],
)

system_message = """
You are a customer support agent. Carefully review the facts about the user below and respond to the user's question.
Be helpful and friendly.
"""

memory = zep.memory.get(session_id=new_session_id)

messages = [
    {
        "role": "system",
        "content": system_message,
    },
    {
        "role": "assistant",
        # The context field is an opinionated string that contains facts and entities relevant to the current conversation.
        "content": memory.context,
    },
    {
        "role": "user",
        "content": alex_message,
    },
]

response = oai_client.chat.completions.create(
    model=os.getenv("OPENAI_API_MODEL"),
    messages=messages,
    temperature=0,
)

print(response.choices[0].message.content)


rich.print(memory.context)

Hi Alex, I'm Amadeus from PaintWiz support. I'm sorry to hear that you're having trouble logging in. I understand you're frustrated with the AI Art Generation feature. Could you please provide more details about the login issue you're experiencing?



FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
<FACTS>
  - user has the id of Alex1800 (2025-03-17 03:28:11 - present)
  - Alex is seeking assistance from PaintWiz support. (2025-03-17 03:28:10 - present)
  - user has the name of Alex Painter (2025-03-17 03:28:11 - present)
  - Amadeus, the assistant, is providing support to Alex. (2025-03-17 03:28:10 - present)
  - user has the email of Alex@painters.com (2025-03-17 03:28:11 - present)
  - Alex1800 is furious about the AI Art Generation, considering it to be completely broken. (2025-03-17 03:28:10 -
present)
</FACTS>

# These are the most relevant entities
# ENTITY_NAME: entity summary
<ENTITIES>
  - User: user
  - PaintWiz support: PaintWiz support is a service platform where users can receive assistance related to PaintWiz
products or services. The assistant, Amadeus, is available to provide support and address user inquiries.
  - Alex@painters.com: user with the email of Alex@painters.com
  - AI Art Generation: The user Alex1800 expressed dissatisfaction with the AI Art Generation, describing it as 
completely broken.
  - Alex1800: Alex1800, a user, expressed dissatisfaction with the AI art generation functionality, describing it 
as completely broken.
  - Alex Painter: Alex Painter encountered issues and sought assistance from PaintWiz support, where Amadeus 
provided help and requested further details about the problem.
  - amadeus (assistant): Amadeus is an assistant providing support for PaintWiz, as demonstrated by its interaction
with Alex, where it requested more details about the issue to offer assistance.
</ENTITIES>

In [12]:
zep.memory.add(
    session_id=new_session_id,
    messages=[
        Message(
            role_type="assistant",
            role=bot_name,
            content=response.choices[0].message.content,
        )
    ],
)


AddMemoryResponse(context=None)

In [16]:
alex_message = "My payment is suspended and I don't know why!"

# We start a new session indicating that Emily has started a new chat with the support agent.
new_session_id = str(uuid.uuid4())

zep.memory.add_session(session_id=new_session_id, user_id=user_id)
# We need to add the Emily's message to the session in order for memory.get to return
# relevant facts related to the message
zep.memory.add(
    session_id=new_session_id,
    messages=[Message(role_type="user", role=user_name, content=alex_message)],
)

system_message = """
You are a customer support agent. Carefully review the facts about the user below and respond to the user's question.
Be helpful and friendly.
"""

memory = zep.memory.get(session_id=new_session_id)

messages = [
    {
        "role": "system",
        "content": system_message,
    },
    {
        "role": "assistant",
        # The context field is an opinionated string that contains facts and entities relevant to the current conversation.
        "content": memory.context,
    },
    {
        "role": "user",
        "content": alex_message,
    },
]

response = oai_client.chat.completions.create(
    model=os.getenv("OPENAI_API_MODEL"),
    messages=messages,
    temperature=0,
)

print(response.choices[0].message.content)


Hi Alex, I understand your payment is suspended and you're not sure why. I can definitely look into this for you.

I see that your account, Alex1800, is currently suspended due to a payment failure. Looking back, a transaction for $99.99 on 2024-09-15 failed because the card ending in 1234 had expired.

To resolve this, you'll need to update your payment information with a valid card. Once you've updated your payment method, the suspension should be lifted automatically. Let me know if you need help with that!



In [17]:
rich.print(memory.context)

FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
<FACTS>
  - The status 'suspended' is due to the reason 'payment failure'. (2025-03-17 03:28:12 - 2025-03-17 03:28:11)
  - The account with ID 'Alex1800' has a status of 'suspended'. (2025-03-17 03:28:11 - 2025-03-17 03:28:12)
  - The account Alex1800 experienced a transaction failure on 2024-08-30 for the amount of 99.99 using a card 
ending in 1234. (2024-08-30 00:00:00 - present)
  - The transaction failure using the card ending in 1234 was due to the card being expired. (2024-08-30 00:00:00 -
2024-09-15 00:00:00)
  - The transaction with status Failed has the failure reason: Card expired. (2024-09-15 00:00:00 - present)
  - The account Alex1800 has a transaction status of Failed. (2024-09-15 00:00:00 - present)
  - The account with ID 'Alex1800' has a transaction amount of 99.99. (2024-07-30 00:00:00 - 2024-09-15 00:00:00)
  - Alex, identified as Alex1800, is experiencing a login issue. (2025-03-17 03:29:09 - 2025-03-17 03:36:14)
  - The account with ID 'Alex1800' has a transaction status of 'Success'. (2024-07-30 00:00:00 - 2024-09-15 
00:00:00)
  - Alex is seeking assistance from PaintWiz support. (2025-03-17 03:28:10 - present)
  - The account with ID 'Alex1800' used a card ending in '1234' for the transaction. (2024-07-30 00:00:00 - 
2024-08-30 00:00:00)
  - user has the email of Alex@painters.com (2025-03-17 03:28:11 - present)
  - Amadeus (assistant) requests details from Alex1800 about the specific tool or feature causing the issue with 
PaintWiz's AI art generation. (2025-03-17 03:28:10 - present)
  - The Magic Pen Tool replaces boat shapes with goats when used. (2024-03-16 14:20:00 - present)
  - user has the id of Alex1800 (2025-03-17 03:28:11 - 2025-03-17 03:29:09)
  - The PaintWiz application incorrectly depicts people with six fingers in Alex1800's drawings. (2025-03-17 
03:28:10 - present)
  - The PaintWiz application incorrectly transforms mountains into fountains in Alex1800's drawings. (2025-03-17 
03:28:10 - present)
  - user has the name of Alex Painter (2025-03-17 03:28:11 - present)
  - Amadeus (assistant) apologizes to Alex1800 for the frustration caused by the issues with PaintWiz's AI art 
generation. (2025-03-17 03:28:10 - present)
  - Amadeus, the assistant, is providing support to Alex. (2025-03-17 03:28:10 - present)
</FACTS>

# These are the most relevant entities
# ENTITY_NAME: entity summary
<ENTITIES>
  - failure_reason: The account 'Alex1800' was suspended on 2024-08-30 due to a payment failure. A subsequent 
transaction of $99.99 on 2024-09-15 was declined because the associated card, ending in '1234', had expired, as 
indicated by the failure reason 'Card expired'.
  - : The user with ID 'Alex1800' has an account status of 'suspended' due to a payment failure.
  - status: The account 'Alex1800' is currently suspended due to a payment failure. This issue arose following a 
failed transaction on 2024-09-15 caused by an expired card, despite another related transaction being marked as 
successful.
  - account_id: The account 'Alex1800' is suspended due to a payment failure. A failed transaction occurred on 
2024-09-15 due to an expired card, despite a prior successful transaction of $99.99 on 2024-07-30 using a card 
ending in 1234.
  - amount: On 2024-09-15, a transaction for $99.99 associated with account ID Alex1800 and card ending in 1234 
failed due to the card being expired.
  - PaintWiz support: PaintWiz, a platform providing AI art generation tools and support services, is addressing a 
user-reported issue. Alex1800 reported problems with the platform's functionality, describing it as 'completely 
broken.' Amadeus, a support representative, apologized for the inconvenience and requested Alex's account ID or 
email address to investigate and resolve the issue.
  - artistic process: The artistic process is men

done!